In [27]:
import duckdb
from pathlib import Path

csv_path = Path("../data/raw/2026-07-08/products.csv.gz")

con = duckdb.connect()
source = f"read_csv('{csv_path}', delim='\t', header=true, sample_size=-1)"

In [28]:
con.execute(f"DESCRIBE SELECT * FROM {source}").fetchall()

[('code', 'VARCHAR', 'YES', None, None, None),
 ('url', 'VARCHAR', 'YES', None, None, None),
 ('creator', 'VARCHAR', 'YES', None, None, None),
 ('created_t', 'BIGINT', 'YES', None, None, None),
 ('created_datetime', 'TIMESTAMP WITH TIME ZONE', 'YES', None, None, None),
 ('last_modified_t', 'BIGINT', 'YES', None, None, None),
 ('last_modified_datetime',
  'TIMESTAMP WITH TIME ZONE',
  'YES',
  None,
  None,
  None),
 ('last_modified_by', 'VARCHAR', 'YES', None, None, None),
 ('last_updated_t', 'BIGINT', 'YES', None, None, None),
 ('last_updated_datetime',
  'TIMESTAMP WITH TIME ZONE',
  'YES',
  None,
  None,
  None),
 ('product_name', 'VARCHAR', 'YES', None, None, None),
 ('abbreviated_product_name', 'VARCHAR', 'YES', None, None, None),
 ('generic_name', 'VARCHAR', 'YES', None, None, None),
 ('quantity', 'VARCHAR', 'YES', None, None, None),
 ('packaging', 'VARCHAR', 'YES', None, None, None),
 ('packaging_tags', 'VARCHAR', 'YES', None, None, None),
 ('packaging_en', 'VARCHAR', 'YES', No

In [34]:
columns = con.execute(f"DESCRIBE SELECT * FROM {source}").fetchall()
print(len(columns))
for col in columns[:10]:
    print(col)

210
('code', 'VARCHAR', 'YES', None, None, None)
('url', 'VARCHAR', 'YES', None, None, None)
('creator', 'VARCHAR', 'YES', None, None, None)
('created_t', 'BIGINT', 'YES', None, None, None)
('created_datetime', 'TIMESTAMP WITH TIME ZONE', 'YES', None, None, None)
('last_modified_t', 'BIGINT', 'YES', None, None, None)
('last_modified_datetime', 'TIMESTAMP WITH TIME ZONE', 'YES', None, None, None)
('last_modified_by', 'VARCHAR', 'YES', None, None, None)
('last_updated_t', 'BIGINT', 'YES', None, None, None)
('last_updated_datetime', 'TIMESTAMP WITH TIME ZONE', 'YES', None, None, None)


In [35]:
candidate_columns = [
    "code",
    "creator",
    "created_datetime",
    "last_modified_datetime",
    "last_updated_datetime",
    "product_name",
    "abbreviated_product_name",
    "generic_name",
    "quantity",
    "packaging",
    "brands",
    "categories",
    "origins",
    "manufacturing_places",
    "labels",
    "emb_codes",
    "cities",
    "purchase_places",
    "stores",
    "countries",
    "ingredients_text",
    "allergens",
    "traces",
    "serving_size",
    "serving_quantity",
    "no_nutrition_data",
    "additives_n",
    "additives",
    "nutriscore_score",
    "nutriscore_grade",
    "nova_group",
    "pnns_groups_1",
    "pnns_groups_2",
    "food_groups",
    "states",
    "brand_owner",
    "environmental_score_score",
    "environmental_score_grade",
    "product_quantity",
    "owner",
    "completeness",
    "main_category",
]

for col_name in candidate_columns:
        result = con.execute(f"""
            SELECT COUNT(*) AS total_rows,
                   COUNT("{col_name}") AS non_null_rows,
                   COUNT(DISTINCT "{col_name}") AS distinct_values
            FROM {source}
        """).fetchone()
        print(col_name, result)

code (15145, 15145, 15145)
creator (15145, 15145, 488)
created_datetime (15145, 15145, 15062)
last_modified_datetime (15145, 15145, 14143)
last_updated_datetime (15145, 15145, 8283)
product_name (15145, 12162, 10239)
abbreviated_product_name (15145, 402, 319)
generic_name (15145, 1172, 391)
quantity (15145, 7133, 1425)
packaging (15145, 1809, 619)
brands (15145, 6831, 2070)
categories (15145, 5580, 1404)
origins (15145, 420, 192)
manufacturing_places (15145, 687, 280)
labels (15145, 2019, 706)
emb_codes (15145, 566, 342)
cities (15145, 0, 0)
purchase_places (15145, 900, 169)
stores (15145, 1529, 335)
countries (15145, 14899, 588)
ingredients_text (15145, 2834, 2406)
allergens (15145, 592, 152)
traces (15145, 73, 60)
serving_size (15145, 403, 133)
serving_quantity (15145, 922, 86)
no_nutrition_data (15145, 328, 4)
additives_n (15145, 2742, 14)
additives (15145, 292, 290)
nutriscore_score (15145, 145, 24)
nutriscore_grade (15145, 8930, 7)
nova_group (15145, 842, 4)
pnns_groups_1 (15145, 

In [38]:
result = con.execute(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE product_name IS NULL AND generic_name IS NULL) AS both_null
    FROM {source}
""").fetchone()
print(result)

(15145, 2978)


In [43]:
result = con.execute(f"""
    SELECT environmental_score_grade, COUNT(*) AS cnt
    FROM {source}
    GROUP BY environmental_score_grade
    ORDER BY cnt DESC
""").fetchall()
print(result)

[('unknown', 9980), (None, 5115), ('d', 17), ('e', 13), ('c', 10), ('f', 6), ('b', 3), ('a', 1)]


In [44]:
result = con.execute(f"""
    SELECT nutriscore_grade, COUNT(*) AS cnt
    FROM {source}
    GROUP BY nutriscore_grade
    ORDER BY cnt DESC
""").fetchall()
print(result)

[('unknown', 7796), (None, 6215), ('not-applicable', 989), ('a', 110), ('b', 12), ('c', 10), ('e', 10), ('d', 3)]


In [39]:
result = con.execute(f"""
    SELECT nova_group, COUNT(*) AS cnt
    FROM {source}
    GROUP BY nova_group
    ORDER BY cnt DESC
""").fetchall()
print(result)

[(None, 14303), (4, 415), (1, 363), (3, 60), (2, 4)]


In [40]:
result = con.execute(f"""
    SELECT pnns_groups_1, COUNT(*) AS cnt
    FROM {source}
    GROUP BY pnns_groups_1
    ORDER BY cnt DESC
""").fetchall()
print(result)

[('unknown', 11118), (None, 3948), ('Fish Meat Eggs', 36), ('Sugary snacks', 14), ('Fat and sauces', 9), ('Composite foods', 6), ('Salty snacks', 4), ('Milk and dairy products', 3), ('Alcoholic beverages', 3), ('Cereals and potatoes', 3), ('Fruits and vegetables', 1)]


In [45]:
result = con.execute(f"""
    SELECT pnns_groups_2, COUNT(*) AS cnt
    FROM {source}
    GROUP BY pnns_groups_2
    ORDER BY cnt DESC
""").fetchall()
print(result)

[('unknown', 11118), (None, 3948), ('Meat', 16), ('Processed meat', 12), ('Biscuits and cakes', 11), ('Fish and seafood', 8), ('Fats', 6), ('One-dish meals', 6), ('Alcoholic beverages', 3), ('Milk and yogurt', 3), ('Dressings and sauces', 3), ('Appetizers', 2), ('Salty and fatty products', 2), ('Sweets', 2), ('Chocolate products', 1), ('Legumes', 1), ('Bread', 1), ('Cereals', 1), ('Vegetables', 1)]


In [41]:
result = con.execute(f"""
    SELECT countries_tags, COUNT(*) AS cnt
    FROM {source}
    GROUP BY countries_tags
    ORDER BY cnt DESC
    LIMIT 15
""").fetchall()
print(result)

[('en:france', 4733), ('en:germany', 1420), ('en:italy', 1099), ('en:spain', 1043), ('en:portugal', 943), ('en:united-states', 911), ('en:indonesia', 500), ('en:united-kingdom', 463), ('en:russia', 331), ('en:belgium', 273), ('en:canada', 246), (None, 246), ('en:poland', 207), ('en:switzerland', 190), ('en:france,en:italy,en:spain', 150)]


In [46]:
result = con.execute(f"""
    SELECT states, COUNT(*) AS cnt
    FROM {source}
    GROUP BY states
    ORDER BY cnt DESC
    LIMIT 10
""").fetchall()
print(result)

[('en:to-be-completed, en:nutrition-facts-to-be-completed, en:ingredients-to-be-completed, en:expiration-date-to-be-completed, en:packaging-code-to-be-completed, en:characteristics-to-be-completed, en:origins-to-be-completed, en:categories-to-be-completed, en:brands-to-be-completed, en:packaging-to-be-completed, en:quantity-to-be-completed, en:product-name-completed, en:photos-to-be-validated, en:packaging-photo-to-be-selected, en:nutrition-photo-to-be-selected, en:ingredients-photo-to-be-selected, en:front-photo-selected, en:photos-uploaded', 1314), ('en:to-be-completed, en:nutrition-facts-to-be-completed, en:ingredients-to-be-completed, en:expiration-date-to-be-completed, en:packaging-code-to-be-completed, en:characteristics-to-be-completed, en:origins-to-be-completed, en:categories-to-be-completed, en:brands-to-be-completed, en:packaging-to-be-completed, en:quantity-completed, en:product-name-completed, en:photos-to-be-validated, en:packaging-photo-to-be-selected, en:nutrition-photo

In [47]:
result = con.execute(f"""
    SELECT MIN(completeness), MAX(completeness), AVG(completeness)
    FROM {source}
""").fetchone()
print(result)

(0.05, 1.0875, 0.3249372392334377)
